# Vastraa — Exploratory Data Analysis
Run `python python/generate_data.py` before executing this notebook.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
ROOT=Path.cwd()
if not (ROOT/'data/raw/orders.csv').exists(): ROOT=ROOT.parent
o=pd.read_csv(ROOT/'data/raw/orders.csv',parse_dates=['order_date'])
o.head()

## Volume, payment mix and city-tier risk

In [ ]:
monthly=o.groupby(o.order_date.dt.to_period('M')).size()
monthly.plot(figsize=(12,4),title='Monthly order volume'); plt.ylabel('Orders'); plt.show()
o.groupby('payment_method').size().sort_values(ascending=False).plot(kind='bar',title='Payment mix'); plt.show()

## Order value distribution and RTO

In [ ]:
o.order_value_inr.clip(upper=o.order_value_inr.quantile(.99)).hist(bins=40,figsize=(8,4)); plt.title('Order value (99th percentile capped for display)'); plt.show()
o.assign(is_rto=o.delivery_outcome.eq('RTO')).groupby(['city_tier','payment_method']).is_rto.mean().unstack().plot(kind='bar',figsize=(9,4),title='RTO rate by tier and payment'); plt.show()

## Data quality profile

In [ ]:
for f in sorted((ROOT/'data/raw').glob('*.csv')):
    d=pd.read_csv(f)
    print(f.name, 'rows=',len(d),'duplicates=',d.duplicated().sum(),'null cells=',d.isna().sum().sum())